# 🚗 Car Fuel Efficiency Prediction — Complete Neural Network Example

This notebook is a **fully worked, self-contained example** of a neural network regression pipeline in PyTorch.
Read it before tackling the exercises — every pattern you need is shown here with a working, runnable example.

> **No download needed** — the dataset is generated synthetically, so this notebook
> runs instantly without a Kaggle account.

---

## 🗺️ Quick Reference — Where to Find What

| What you need to write in the exercises | Where it is in this notebook |
|-----------------------------------------|------------------------------|
| `nn.Linear(in, out)` + `nn.ReLU()` — adding layers | **§3 Architecture** |
| `nn.MSELoss()` — loss function | **§4a Training Setup** |
| `torch.optim.Adam(model.parameters(), lr=..., weight_decay=...)` | **§4a Training Setup** |
| Early stopping: update `best_val_loss`, reset `patience_counter`, `break` | **§4b Training Loop** |
| `df['new_col'] = df['col_a'] / df['col_b']` — feature engineering | **§2a Feature Engineering** |
| `np.log(df['col'].values).reshape(-1, 1)` — log-transform the target | **§2b Log-Transform** |
| `TensorDataset(X_t, y_t)` + `DataLoader(..., batch_size=256, shuffle=True)` | **§2d DataLoader** |
| `ReduceLROnPlateau(optimizer, mode='min', factor=..., patience=...)` | **§4a Training Setup** |

---

## What this notebook predicts

We predict the **fuel efficiency (MPG — miles per gallon)** of a car from its physical and technical
characteristics. The goal is identical to the exercises — a **regression task** — just with a different
domain and a different network architecture.

| Feature | Description |
|---------|-------------|
| `cylinders` | Number of engine cylinders (4, 6, or 8) |
| `displacement_L` | Engine displacement in litres |
| `horsepower` | Engine power (HP) |
| `weight_kg` | Car weight in kg |
| `acceleration` | Time to reach 100 km/h (seconds) |
| `model_year` | Year the car was manufactured |
| `is_hybrid` | 1 if hybrid, 0 otherwise |
| `drag_coeff` | Aerodynamic drag coefficient |
| `mpg` | Fuel efficiency — **this is what we predict** |

## Neural network architecture (different from the exercises)

```
  Input (11 features) → 256 → 128 → 64 → 32 → 1
```

The exercises use `4 → 128 → 128 → 64 → 1`. This network is **wider** (256 in the first layer)
and **deeper** (4 hidden layers instead of 3). The **pattern** for each layer is exactly the same.


---

## Step 0 — Install & Import

We import everything from `utils.py` with a single line — exactly as in the exercise notebooks.
All libraries (`torch`, `numpy`, `sklearn`, etc.) and all helper functions come from there.

> Open `utils.py` at any time to see the full implementation of any helper function.


In [ ]:
!git clone -b week1 https://github.com/eth-bmai-fs26/coding-exercises.git
!git fetch && git checkout week1
%cd coding-exercises/cx3-intro-to-nns/

In [ ]:
%pip install torch scikit-learn pandas numpy matplotlib seaborn --quiet
print('All packages installed.')

In [ ]:
from utils import *

print('Everything imported successfully!')

# What is now available (same as in the exercise notebooks):
#   torch, nn, TensorDataset, DataLoader          — PyTorch
#   np, pd                                         — NumPy and Pandas
#   train_test_split, StandardScaler               — scikit-learn
#   plt, sns                                       — matplotlib and seaborn
#
#   generate_car_dataset()                         — generate the synthetic car data
#   plot_car_dataset(df)                           — 4 exploration charts
#   split_and_scale_data_tricks(feat, log_y, orig) — 80/10/10 split + feature scaling
#   convert_to_tensors(...)                        — NumPy arrays → float32 tensors
#   run_epoch_minibatch(...)                       — one mini-batch training epoch
#   plot_training_history(train_l, val_l)          — loss curves
#   get_predictions_log(model, test_X)             — exp() of log-target predictions
#   print_performance_metrics(actual, pred, ...)   — MAE / RMSE / MAPE
#   plot_predictions(actual, pred, ...)            — scatter, residuals, bar chart

---

## §1 — Data Generation & Exploration

generate_car_dataset() creates 3,000 synthetic cars using NumPy (see `utils.py` for the formula).
It returns a pandas DataFrame — exactly the same type that `load_dataset()` returns in the exercises.

`plot_car_dataset(df)` shows 4 exploration charts: target distribution, feature correlations,
and two scatter plots for the strongest predictors.


In [ ]:
df = generate_car_dataset()
df.head()

In [ ]:
plot_car_dataset(df)

---

## §2 — Data Preparation

### §2a — Feature Engineering

We add 3 new columns that capture **interactions** between existing features.
A neural network can theoretically learn these, but giving them pre-computed values
makes training faster and more reliable.

| New feature | Formula | Physical meaning |
|-------------|---------|------------------|
| `hp_per_cyl` | `horsepower / cylinders` | Power density — 50 HP/cyl differs from 25 HP/cyl even at the same total HP |
| `weight_to_power` | `weight_kg / horsepower` | Weight-to-power ratio — heavier and underpowered = worse MPG |
| `displacement_sq` | `displacement_L ** 2` | Engine size has a non-linear effect on consumption |

**Pattern used in the exercises (Trick 2 — Feature Engineering):**

```python
df['new_column']  = df['col_a'] / df['col_b']   # division
df['new_column2'] = df['col_a'] * df['col_b']   # multiplication
df['new_column3'] = df['col_a'] ** 2            # squaring
```


In [ ]:
df['hp_per_cyl']      = df['horsepower'] / df['cylinders']    # power density
df['weight_to_power'] = df['weight_kg']  / df['horsepower']   # weight-to-power ratio
df['displacement_sq'] = df['displacement_L'] ** 2             # non-linear engine size

print(f"Original features:    8")
print(f"Engineered features:  3")
print(f"Total features:       {len(df.columns) - 1}   (all columns except 'mpg')")

### §2b — Separate Features & Target + Log-Transform

Look at the MPG histogram above — it has a **right skew**: most cars cluster at 20–35 MPG,
but a few efficient models reach 50+. MSE on raw MPG penalises absolute errors equally,
even though a 3 MPG mistake on a 15 MPG car (20%) is far worse than on a 45 MPG car (7%).

Applying `np.log()` fixes this: MSE on log-MPG approximates a **percentage error**.
At prediction time we reverse it with `np.exp()`.

**Pattern used in the exercises (Trick 1 — Log-Transform):**

```python
log_target = np.log(df['target_column'].values).reshape(-1, 1)
#                                                  ^^^^^^^^^^^
#            turns shape (N,) into shape (N, 1) — PyTorch needs a 2-D column vector
```


In [ ]:
feature_columns = [c for c in df.columns if c != 'mpg']
features        = df[feature_columns].values               # shape (3000, 11)

# Log-transform the target
log_mpg      = np.log(df['mpg'].values).reshape(-1, 1)    # shape (3000, 1)
original_mpg = df['mpg'].values.reshape(-1, 1)             # kept for final metrics in real MPG

print(f"Feature matrix shape:  {features.shape}")
print(f"Log-target shape:      {log_mpg.shape}")
print(f"\nOriginal MPG range:  {original_mpg.min():.1f} – {original_mpg.max():.1f}")
print(f"Log-MPG range:       {log_mpg.min():.2f} – {log_mpg.max():.2f}  (much smaller!)")

# Sanity check: exp(log(x)) == x
print(f"\nSanity check — log then exp: {np.exp(log_mpg[0, 0]):.2f} ≈ {original_mpg[0, 0]:.2f}")

### §2c — Train / Validation / Test Split + Feature Scaling

`split_and_scale_data_tricks()` handles the 80 / 10 / 10 split and fits a `StandardScaler`
on the training features only (to avoid data leakage).
It returns 8 NumPy arrays — see `utils.py` for the details.

| Set | Size | Role |
|-----|------|------|
| Training | 80 % | The model **learns** from this data |
| Validation | 10 % | Used to **monitor** training and detect overfitting |
| Test | 10 % | **Final evaluation** on data the model has never seen |


In [ ]:
(train_features, val_features, test_features,
 train_log,      val_log,      test_log,
 val_orig,       test_orig) = split_and_scale_data_tricks(features, log_mpg, original_mpg)

### §2d — Convert to PyTorch Tensors + Create a DataLoader

`convert_to_tensors()` turns the six NumPy arrays into `torch.float32` tensors.

We then wrap the training tensors in a `DataLoader` so the loop receives them in
**shuffled mini-batches of 256** rather than all at once.
Each batch gives a cheaper, noisier gradient estimate, which acts as implicit regularization.

**Pattern used in the exercises (Trick 3 — DataLoader):**

```python
# Step 1 — pair features and targets so they stay in sync when shuffled
train_dataset = TensorDataset(train_features_t, train_targets_t)

# Step 2 — wrap in a DataLoader that yields mini-batches
train_loader  = DataLoader(train_dataset, batch_size=256, shuffle=True)
```


In [ ]:
# Convert NumPy arrays to PyTorch float32 tensors
(train_features_t, val_features_t, test_features_t,
 train_log_t,      val_log_t,      test_log_t) = convert_to_tensors(
    train_features, val_features, test_features,
    train_log, val_log, test_log
)

# Create the DataLoader for mini-batch training
train_dataset = TensorDataset(train_features_t, train_log_t)              # pair features ↔ targets
train_loader  = DataLoader(train_dataset, batch_size=256, shuffle=True)  # shuffle each epoch

print(f"Tensor shapes — train: {train_features_t.shape},  val: {val_features_t.shape},  test: {test_features_t.shape}")
print(f"DataLoader: {len(train_loader)} batches × up to 256 samples per epoch")

---

## §3 — Neural Network Architecture

The model is a Python class that inherits from `nn.Module`.
All layers live inside `nn.Sequential`, which chains them automatically.

```
  ┌────────────┐   ┌──────────┐   ┌──────────┐   ┌─────────┐   ┌─────────┐   ┌──────────┐
  │   INPUT    │   │  L1      │   │  L2      │   │  L3     │   │  L4     │   │  OUTPUT  │
  │ 11 features│──▶│ 256 neu. │──▶│ 128 neu. │──▶│ 64 neu. │──▶│ 32 neu. │──▶│ 1 neuron │
  └────────────┘   │ Linear   │   │ Linear   │   │ Linear  │   │ Linear  │   │ Linear   │
                   │ ReLU     │   │ ReLU     │   │ ReLU    │   │ ReLU    │   │ (no act.)│
                   └──────────┘   └──────────┘   └─────────┘   └─────────┘   └──────────┘
```

**Pattern for each hidden layer — used in both exercise notebooks:**

```python
nn.Linear(in_features, out_features),   # weighted sum: output = W · input + b
nn.ReLU(),                               # activation:   max(0, x)
```

The **output layer** has no activation — for regression we want a raw number, not a probability.

> `_init_weights` applies **Kaiming (He) initialization**, designed for ReLU activations.
> It keeps activation variances stable across layers to prevent vanishing or exploding gradients.


In [ ]:
class CarMPGModel(nn.Module):
    """
    Fully-connected network for predicting car fuel efficiency.

    Architecture:  Input(11) -> 256 -> 128 -> 64 -> 32 -> 1
    Activation:    ReLU after each hidden layer (no activation on output).
    Init:          Kaiming normal — optimised for ReLU networks.
    """

    def __init__(self, n_features):
        super().__init__()

        self.network = nn.Sequential(
            # Hidden layer 1: n_features -> 256
            nn.Linear(n_features, 256),
            nn.ReLU(),

            # Hidden layer 2: 256 -> 128
            nn.Linear(256, 128),
            nn.ReLU(),

            # Hidden layer 3: 128 -> 64
            nn.Linear(128, 64),
            nn.ReLU(),

            # Hidden layer 4: 64 -> 32
            nn.Linear(64, 32),
            nn.ReLU(),

            # Output layer: 32 -> 1  (predicted log-MPG; no activation)
            nn.Linear(32, 1)
        )

        self._init_weights()

    def _init_weights(self):
        """Kaiming (He) initialization for all Linear layers."""
        for module in self.network:
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, nonlinearity='relu')
                nn.init.zeros_(module.bias)

    def forward(self, x):
        """Push the input tensor through every layer in sequence."""
        return self.network(x)


# Instantiate: always pass the number of input features
model = CarMPGModel(n_features=train_features_t.shape[1])

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal trainable parameters: {total_params:,}")

---

## §4 — Training

### §4a — Loss Function, Optimizer, and Scheduler

Three objects control how learning happens:

| Object | Class | Role |
|--------|-------|------|
| `loss_fn` | `nn.MSELoss()` | Measures how wrong predictions are — average of (pred − target)² |
| `optimizer` | `torch.optim.Adam(...)` | Updates the weights to reduce the loss |
| `scheduler` | `ReduceLROnPlateau(...)` | Reduces the learning rate when progress stalls |

**Hyperparameters:**

| Name | Value | Meaning |
|------|-------|---------|
| `learning_rate` | 0.001 | Initial step size for weight updates |
| `weight_decay` | 1e-4 | L2 regularization — penalizes large weights |
| `scheduler factor` | 0.5 | LR is multiplied by this on each plateau (`new_lr = old_lr × 0.5`) |
| `scheduler patience` | 50 | Epochs to wait before reducing LR |

**Pattern used in both exercise notebooks:**

```python
loss_fn   = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=50
)
```


In [ ]:
loss_fn          = nn.MSELoss()    # Mean Squared Error: average of (prediction - target)^2

learning_rate    = 0.001
weight_decay     = 1e-4
number_of_epochs = 2000

# Adam adapts the learning rate individually for each parameter.
# model.parameters() gives Adam access to all tensors that need updating.
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)

# Halves the LR whenever the validation loss does not improve for 50 epochs.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',    # we are MINIMISING the loss
    factor=0.5,    # new_lr = old_lr * 0.5
    patience=50    # wait 50 epochs before reducing
)

print(f"Loss:       {loss_fn}")
print(f"Optimizer:  Adam  (lr={learning_rate}, weight_decay={weight_decay})")
print(f"Scheduler:  ReduceLROnPlateau  (factor=0.5, patience=50)")
print(f"Max epochs: {number_of_epochs}")

### §4b — Training Loop

`run_epoch_minibatch()` handles the boilerplate of one epoch (mini-batch forward/backward pass
on the training set + full validation pass). It returns `(train_loss, val_loss)` as plain floats.

The three things you manage yourself — and that students fill in the exercises — are:
- calling `scheduler.step(val_loss)` after each epoch
- implementing **early stopping** (the three lines below)

**Early stopping pattern:**

```python
if val_loss < best_val_loss:
    best_val_loss    = val_loss    # remember the new best
    patience_counter = 0           # reset the stagnation counter
    best_model_state = ...         # save the model weights
else:
    patience_counter += 1          # one more epoch without improvement

if patience_counter >= patience:
    print(f"Early stopping at epoch {epoch}")
    break                          # exit the for-loop
```


In [ ]:
train_losses = []
val_losses   = []

best_val_loss    = float('inf')   # start with "infinitely bad"
patience_counter = 0
patience         = 100
best_model_state = None

for epoch in range(number_of_epochs):

    # run_epoch_minibatch handles: mini-batch loop, backward pass, validation
    train_loss, val_loss = run_epoch_minibatch(
        model, optimizer, loss_fn,
        train_loader, val_features_t, val_log_t
    )

    # Tell the scheduler the current val_loss so it can reduce LR if needed
    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # --- Early stopping ---
    if val_loss < best_val_loss:
        best_val_loss    = val_loss              # new best — save it
        patience_counter = 0                     # reset stagnation counter
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1                    # no improvement — count it

    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch} — no improvement for {patience} epochs")
        break

    if epoch % 100 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:>4d} | Train: {train_loss:.4f} | "
              f"Val: {val_loss:.4f} | LR: {current_lr:.6f}")

if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\nLoaded best model  (val_loss = {best_val_loss:.4f})")

print(f"\nTraining complete — total epochs run: {epoch + 1}")

In [ ]:
plot_training_history(train_losses, val_losses)

---

## §5 — Model Evaluation

`get_predictions_log()` runs the model on the test set and applies `np.exp()` to convert
log-MPG predictions back to real MPG — the exact reverse of the log-transform in §2b.

`print_performance_metrics()` prints MAE, RMSE, MAPE and a per-quartile breakdown.
We pass `unit='MPG'` so the printed values use the right unit label instead of `$`.


In [17]:
test_predictions = get_predictions_log(model, test_features_t)

In [ ]:
print_performance_metrics(test_orig, test_predictions, show_r2=True, unit='MPG')

---

## §6 — Prediction Visualizations

`plot_predictions()` shows 3 charts: predicted vs actual scatter, residuals histogram,
and a bar chart of 20 random samples. We pass `target_name` and `unit` to get
correct axis labels for MPG instead of the default house-price labels.


In [ ]:
plot_predictions(test_orig, test_predictions,
                 target_name='Fuel Efficiency', unit='(MPG)')

---

## Summary — Full Pipeline at a Glance

```
 Raw data (DataFrame)
      │
      ├─ §2a  Feature engineering       df['new'] = df['a'] / df['b']             ← write this yourself
      ├─ §2b  Log-transform target      log_y = np.log(df['col']).reshape(-1, 1)   ← write this yourself
      ├─ §2c  Split + StandardScaler    split_and_scale_data_tricks(...)            ← utils.py
      ├─ §2d  Tensor conversion         convert_to_tensors(...)                     ← utils.py
      ├─ §2d  DataLoader                TensorDataset + DataLoader                 ← write this yourself
      │
 PyTorch tensors + DataLoader
      │
      ├─ §3   nn.Sequential(nn.Linear, nn.ReLU, ...)                               ← write this yourself
      │
      ├─ §4a  nn.MSELoss()                                                          ← write this yourself
      │        torch.optim.Adam(model.parameters(), lr=..., weight_decay=...)
      │        ReduceLROnPlateau(optimizer, mode='min', factor=..., patience=...)
      │
      ├─ §4b  Training loop
      │         for epoch in range(N):
      │           run_epoch_minibatch(...)          ← utils.py
      │           scheduler.step(val_loss)          ← write this yourself
      │           early stopping logic              ← write this yourself
      │
      ├─ §5   plot_training_history(...)            ← utils.py
      │        get_predictions_log(...)             ← utils.py
      │        print_performance_metrics(...)       ← utils.py
      └─ §6   plot_predictions(...)                 ← utils.py
```

### Connection to the exercises

| Exercise notebook | Concepts shown here |
|-------------------|---------------------------------|
| `nn_base_exercise.ipynb` | §3 layer definition, §4a loss + optimizer, §4b early stopping |
| `nn_tricks_exercise.ipynb` | §2a feature engineering, §2b log-transform, §2d DataLoader, §4a scheduler |
